<a href="https://colab.research.google.com/github/tobiasllop/Tesis/blob/main/Tesis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Tesis 2026 - Tobias Llop

In [2]:
# import Libraries
import pandas as pd
import numpy as np
import matplotlib as plt
import pyarrow.parquet as pq
from google.colab import drive
# Install xlsxwriter if not already installed
!pip install xlsxwriter

drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
###############################################################################
#### DATA LOADING Deudores Juridicas #############################################################
#################################################################################

file_path = '/content/drive/MyDrive/Tesis2026/deudores_final_febrero.parquet'
prefijos_fisicas = ('20', '23', '24', '27')

# Filtrado dinámico de TODO el dataset
parquet_file = pq.ParquetFile(file_path)
filtered_chunks = []
total_rows_scanned = 0

print("Iniciando el procesamiento del dataset completo...")
print("Esto puede tomar unos minutos dependiendo del tamaño del archivo.")

# Iteramos por todo el archivo sin límite de filas (target_rows)
for batch in parquet_file.iter_batches(batch_size=200000):
    chunk_df = batch.to_pandas()
    total_rows_scanned += len(chunk_df)

    # Asegurar tipos para el filtrado
    chunk_df['tipo_id'] = chunk_df['tipo_id'].astype(str)
    chunk_df['nro_id'] = chunk_df['nro_id'].astype(str)

    # Aplicar filtro de exclusión (Personas Jurídicas)
    is_fisica = (chunk_df['tipo_id'] == '11') & (chunk_df['nro_id'].str.startswith(prefijos_fisicas))
    juridicas_chunk = chunk_df[~is_fisica]

    if not juridicas_chunk.empty:
        filtered_chunks.append(juridicas_chunk)

    # Feedback visual del progreso
    if total_rows_scanned % 1000000 == 0:
        print(f"Filas procesadas: {total_rows_scanned}...")

# Combinar todos los resultados filtrados
if filtered_chunks:
    df = pd.concat(filtered_chunks)
    print(f"¡Procesamiento completado!")
    print(f"Total de filas analizadas: {total_rows_scanned}")
    print(f"Total de 'personas jurídicas' encontradas: {len(df)}")
    display(df.head())
else:
    print("No se encontraron 'personas jurídicas' en todo el dataset.")

Iniciando el procesamiento del dataset completo...
Esto puede tomar unos minutos dependiendo del tamaño del archivo.
Filas procesadas: 1000000...
Filas procesadas: 2000000...
Filas procesadas: 3000000...
Filas procesadas: 4000000...
Filas procesadas: 5000000...
Filas procesadas: 6000000...
Filas procesadas: 7000000...
Filas procesadas: 8000000...
Filas procesadas: 9000000...
Filas procesadas: 10000000...
Filas procesadas: 11000000...
Filas procesadas: 12000000...
Filas procesadas: 13000000...
Filas procesadas: 14000000...
Filas procesadas: 15000000...
Filas procesadas: 16000000...
Filas procesadas: 17000000...
Filas procesadas: 18000000...
Filas procesadas: 19000000...
Filas procesadas: 20000000...
Filas procesadas: 21000000...
Filas procesadas: 22000000...
Filas procesadas: 23000000...
Filas procesadas: 24000000...
Filas procesadas: 25000000...
Filas procesadas: 26000000...
Filas procesadas: 27000000...
Filas procesadas: 28000000...
Filas procesadas: 29000000...
Filas procesadas: 3000

,cod_entidad,fecha,tipo_id,nro_id,actividad,situacion,prestamos,sin_uso,garantias_otorg,otros_conceptos,...,contragar_pref_B,sin_contragar_pref,previsiones,deuda_cubierta,proceso_judicial,refinanciaciones,recat_obligatoria,sit_juridica,irrecuperable,dias_atraso
84744,00007,202602,11,30500117571,182.0,1,16287.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0,0,0,0,0,0,0.0
84816,00007,202602,11,30500781293,152.0,1,3563669.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0,0,0,0,0,0,0.0
84888,00007,202602,11,30502921653,282.0,1,82.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0,0,0,0,0,0,0.0
84960,00007,202602,11,30503272217,682.0,1,60.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0,0,0,0,0,0,0.0
85032,00007,202602,11,30503746235,464.0,1,147421.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0,0,0,0,0,0,0.0


In [4]:
###############################################################################
#### DATA LOADING Entidades #############################################################
#################################################################################



file_path_entidades = '/content/drive/MyDrive/Tesis2026/EntidadesFinancierasBCRA.csv'
df_entidades = pd.read_csv(file_path_entidades)

# Seleccionamos solo las columnas necesarias y nos aseguramos de que no haya duplicados
df_entidades_selected = df_entidades[['cod_entidad', 'nombre_entidad', 'grupo_entidad']].copy()

# Limpiamos cod_entidad
df['cod_entidad'] = df['cod_entidad'].astype(str).str.zfill(5)
df_entidades_selected['cod_entidad'] = df_entidades_selected['cod_entidad'].astype(str).str.zfill(5)

# Eliminamos columnas si ya existen en df para evitar sufijos _x _y
columns_to_add = ['nombre_entidad', 'grupo_entidad']
df = df.drop(columns=[c for c in columns_to_add if c in df.columns])


In [5]:
#################################################################################
####### Data Loading Personas Juridicas #########################################
##################################################################################

# Define the path to the padron_juridicas.parquet file
file_path_padron = '/content/drive/MyDrive/Tesis2026/padron_juridicas.parquet'

# Load the parquet file into a pandas DataFrame
df_padron_juridicas = pd.read_parquet(file_path_padron)

print("DataFrame 'df_padron_juridicas' loaded successfully:")
display(df_padron_juridicas.head())
print(f"Total rows in df_padron_juridicas: {len(df_padron_juridicas)}")

DataFrame 'df_padron_juridicas' loaded successfully:


,cuit,denominacion,actividad,marca_baja,cuit_reemplazo,fecha_nacimiento,sexo,codigo_postal,provincia,fecha_fallecimiento
0,30717086291,OCULARYB S.A.S.,869090,,NaN,2020-08-12,,4107,14,
1,30618725010,SULKY SOCIEDAD ANONIMA,702000,,0.0,1979-09-03,,1416,0,
2,30697847940,VIVANCO PEDRO Y SAAVEDRA CARLOS SH,631035,,NaN,1998-07-03,,8300,20,
3,30617991876,GOMEZ Y AGUIRRE S H,619094,,0.0,1901-01-01,,2942,1,
4,30584627170,LIGA DEPARTAMENTAL DE FUTBOL DEL DEPTO.SAN MARTIN,949990,,NaN,1938-04-02,,2452,12,


Total rows in df_padron_juridicas: 1837948


In [6]:
##################################################################################
########### Data Manipulation ##################################################
###############################################################################

# Ensure types for merge keys are consistent
df['cod_entidad'] = df['cod_entidad'].astype(str).str.zfill(5)
# df_entidades_selected['cod_entidad'] is already handled in its loading cell L1eFS31ZG_sg.

df['nro_id'] = df['nro_id'].astype(str)


# 1. Merge with df_entidades_selected
# Assuming 'columns_to_add' is already defined in a previous cell as ['nombre_entidad', 'grupo_entidad']
# and that any pre-existing 'nombre_entidad'/'grupo_entidad' in 'df' were dropped by the previous cell's code.
df_current = pd.merge(df, df_entidades_selected, on='cod_entidad', how='left')


# 2. Merge with df_padron_juridicas
cols_from_padron_to_merge = [] # Columns from df_padron_juridicas to select for the merge
cols_to_insert_after_nro_id = [] # Columns from padron that will be reordered after 'nro_id'

if 'df_padron_juridicas' in globals() and not df_padron_juridicas.empty:
    # Ensure 'cuit' from df_padron_juridicas is string for merging
    df_padron_juridicas['cuit'] = df_padron_juridicas['cuit'].astype(str)

    # The right_on key is 'cuit'. We definitely need to select it.
    if 'cuit' in df_padron_juridicas.columns:
        cols_from_padron_to_merge.append('cuit')
        # If 'cuit' is not already in df_current, add it to the reordering list
        if 'cuit' not in df_current.columns:
            cols_to_insert_after_nro_id.append('cuit')

    # Now, find other columns from df_padron_juridicas to add, excluding 'nro_id', 'tipo_id', and those already in df_current
    for col in df_padron_juridicas.columns:
        if col not in ['nro_id', 'tipo_id', 'cuit'] and col not in df_current.columns:
            cols_from_padron_to_merge.append(col)
            cols_to_insert_after_nro_id.append(col)

    if cols_from_padron_to_merge:
        df_current = pd.merge(df_current, df_padron_juridicas[cols_from_padron_to_merge],
                              left_on='nro_id', right_on='cuit', how='left')
    else:
        print("No new unique columns found in df_padron_juridicas to merge. Skipping padron merge for additional columns.")
else:
    print("Warning: df_padron_juridicas was not loaded successfully or is empty. Skipping merge with padron data.")


# 3. Reorder columns according to user's request
final_order = []
seen_cols_in_final_order = set()

# Columns that should appear after 'cod_entidad'
entity_cols_to_position = ['nombre_entidad', 'grupo_entidad']
# Columns that should appear after 'nro_id' are in `cols_to_insert_after_nro_id`.

all_cols_after_merges = df_current.columns.tolist()

for col in all_cols_after_merges:
    if col not in seen_cols_in_final_order: # Add if not already placed
        final_order.append(col)
        seen_cols_in_final_order.add(col)

        if col == 'cod_entidad':
            for entity_col in entity_cols_to_position:
                if entity_col in all_cols_after_merges and entity_col not in seen_cols_in_final_order:
                    final_order.append(entity_col)
                    seen_cols_in_final_order.add(entity_col)
        elif col == 'nro_id':
            for padron_col in cols_to_insert_after_nro_id:
                if padron_col in all_cols_after_merges and padron_col not in seen_cols_in_final_order:
                    final_order.append(padron_col)
                    seen_cols_in_final_order.add(padron_col)

# Assign the reordered DataFrame back to `df`
df = df_current[final_order]

print("DataFrame actualizado con nombres de entidades y padrón en sus posiciones especificadas:")
display(df.head())
display(df.info())

DataFrame actualizado con nombres de entidades y padrón en sus posiciones especificadas:


,cod_entidad,nombre_entidad,grupo_entidad,fecha,tipo_id,nro_id,cuit,denominacion,marca_baja,cuit_reemplazo,...,contragar_pref_B,sin_contragar_pref,previsiones,deuda_cubierta,proceso_judicial,refinanciaciones,recat_obligatoria,sit_juridica,irrecuperable,dias_atraso
0,00007,BANCO DE GALICIA Y BUENOS AIRES S.A.,Banco,202602,11,30500117571,30500117571,ASOCIACION CASA EDITORA SUDAMERICANA,,NaN,...,0.0,0.0,0.0,0,0,0,0,0,0,0.0
1,00007,BANCO DE GALICIA Y BUENOS AIRES S.A.,Banco,202602,11,30500781293,30500781293,GRIMOLDI SOCIEDAD ANONIMA,,NaN,...,0.0,0.0,0.0,0,0,0,0,0,0,0.0
2,00007,BANCO DE GALICIA Y BUENOS AIRES S.A.,Banco,202602,11,30502921653,30502921653,MAIZCO SOCIEDAD ANONIMA INDUSTRIAL Y COMERCIAL,,NaN,...,0.0,0.0,0.0,0,0,0,0,0,0,0.0
3,00007,BANCO DE GALICIA Y BUENOS AIRES S.A.,Banco,202602,11,30503272217,30503272217,CONSORCIO DE PROPIETARIOS H. YRIGOYEN 2147,,NaN,...,0.0,0.0,0.0,0,0,0,0,0,0,0.0
4,00007,BANCO DE GALICIA Y BUENOS AIRES S.A.,Banco,202602,11,30503746235,30503746235,LABORATORIOS FERRING SOCIEDAD ANONIMA,,NaN,...,0.0,0.0,0.0,0,0,0,0,0,0,0.0


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 477801 entries, 0 to 477800
Data columns (total 35 columns):
 #   Column               Non-Null Count   Dtype  
---  ------               --------------   -----  
 0   cod_entidad          477801 non-null  object 
 1   nombre_entidad       477801 non-null  object 
 2   grupo_entidad        477801 non-null  object 
 3   fecha                477801 non-null  object 
 4   tipo_id              477801 non-null  object 
 5   nro_id               477801 non-null  object 
 6   cuit                 477694 non-null  object 
 7   denominacion         477694 non-null  object 
 8   marca_baja           477694 non-null  object 
 9   cuit_reemplazo       12000 non-null   float64
 10  fecha_nacimiento     477694 non-null  object 
 11  sexo                 477694 non-null  object 
 12  codigo_postal        477694 non-null  object 
 13  provincia            477694 non-null  float64
 14  fecha_fallecimiento  477694 non-null  object 
 15  actividad        

None

In [12]:
print(df.head())

  cod_entidad                        nombre_entidad grupo_entidad   fecha  \
0       00007  BANCO DE GALICIA Y BUENOS AIRES S.A.         Banco  202602   
1       00007  BANCO DE GALICIA Y BUENOS AIRES S.A.         Banco  202602   
2       00007  BANCO DE GALICIA Y BUENOS AIRES S.A.         Banco  202602   
3       00007  BANCO DE GALICIA Y BUENOS AIRES S.A.         Banco  202602   
4       00007  BANCO DE GALICIA Y BUENOS AIRES S.A.         Banco  202602   

  tipo_id       nro_id         cuit  \
0      11  30500117571  30500117571   
1      11  30500781293  30500781293   
2      11  30502921653  30502921653   
3      11  30503272217  30503272217   
4      11  30503746235  30503746235   

                                     denominacion marca_baja  cuit_reemplazo  \
0            ASOCIACION CASA EDITORA SUDAMERICANA                        NaN   
1                       GRIMOLDI SOCIEDAD ANONIMA                        NaN   
2  MAIZCO SOCIEDAD ANONIMA INDUSTRIAL Y COMERCIAL             

In [7]:
output_path = '/content/drive/MyDrive/Tesis2026/'

# Prepare list to store top 10 dataframes for export
top_10_dfs = {}

# Define the single Excel file path
single_excel_filename = f'{output_path}all_situaciones_top10.xlsx'

# Create an Excel writer object
with pd.ExcelWriter(single_excel_filename, engine='xlsxwriter') as writer:
    for i in range(1, 6):
        # Filter dataframe by 'situacion'
        current_df = df[df['situacion'] == str(i)].copy()

        # Ensure 'prestamos' is numeric and fill NaNs with 0
        current_df['prestamos'] = pd.to_numeric(current_df['prestamos'], errors='coerce').fillna(0)

        # Sort by 'prestamos' in descending order and get the top 10
        df_situacion_top_10 = current_df.sort_values(by='prestamos', ascending=False).head(10)

        # Store the top 10 dataframe (optional, but good for inspection if needed later)
        top_10_dfs[f'df_situacion_{i}'] = df_situacion_top_10

        print(f"\nTop 10 de df_situacion_{i} (ordenado por prestamos descendente):")
        display(df_situacion_top_10)

        # Export to a specific sheet in the single Excel file
        sheet_name = f'df_situacion_{i}'
        df_situacion_top_10.to_excel(writer, sheet_name=sheet_name, index=False)
        print(f"Exportado df_situacion_{i}_top10 a la hoja '{sheet_name}' en {single_excel_filename}")

print(f"\n¡Proceso completado! Todos los dataframes top 10 han sido exportados a un único archivo Excel: {single_excel_filename}")


Top 10 de df_situacion_1 (ordenado por prestamos descendente):


,cod_entidad,nombre_entidad,grupo_entidad,fecha,tipo_id,nro_id,cuit,denominacion,marca_baja,cuit_reemplazo,...,contragar_pref_B,sin_contragar_pref,previsiones,deuda_cubierta,proceso_judicial,refinanciaciones,recat_obligatoria,sit_juridica,irrecuperable,dias_atraso
144287,00011,BANCO DE LA NACION ARGENTINA,Banco,202602,11,30546676427,30546676427,TESORERIA GENERAL DE LA NACION,,NaN,...,0.0,0.0,0.0,0,0,0,0,0,0,0.0
136995,00011,BANCO DE LA NACION ARGENTINA,Banco,202602,11,30546689979,30546689979,YPF SOCIEDAD ANONIMA,,NaN,...,0.0,0.0,0.0,0,0,0,0,0,0,0.0
39034,00007,BANCO DE GALICIA Y BUENOS AIRES S.A.,Banco,202602,11,30678519681,30678519681,FORD ARGENTINA SOCIEDAD EN COMANDITA POR ACCIONES,,NaN,...,0.0,0.0,0.0,0,0,0,0,0,0,0.0
112843,00007,BANCO DE GALICIA Y BUENOS AIRES S.A.,Banco,202602,11,33650305499,33650305499,CENTRAL PUERTO S A,,NaN,...,0.0,0.0,0.0,0,0,0,0,0,0,0.0
152878,00011,BANCO DE LA NACION ARGENTINA,Banco,202602,11,30695542476,30695542476,"PAN AMERICAN ENERGY, S.L., SUCURSAL ARGENTINA",,NaN,...,0.0,0.0,0.0,0,0,0,0,0,0,0.0
230234,00017,BANCO BBVA ARGENTINA S.A.,Banco,202602,11,30703088534,30703088534,MERCADOLIBRE S.R.L.,,NaN,...,0.0,0.0,0.0,0,0,0,0,0,0,0.0
8766,00007,BANCO DE GALICIA Y BUENOS AIRES S.A.,Banco,202602,11,30506726804,30506726804,RAIZEN ARGENTINA S.A.U.,,NaN,...,0.0,0.0,0.0,0,0,0,0,0,0,0.0
276986,00034,BANCO PATAGONIA S.A.,Banco,202602,11,30703088534,30703088534,MERCADOLIBRE S.R.L.,,NaN,...,0.0,0.0,0.0,0,0,0,0,0,0,0.0
201186,00015,INDUSTRIAL AND COMMERCIAL BANK OF CHINA (ARGEN...,Banco,202602,11,30695542476,30695542476,"PAN AMERICAN ENERGY, S.L., SUCURSAL ARGENTINA",,NaN,...,0.0,333.0,0.0,0,0,0,0,0,0,0.0
217242,00017,BANCO BBVA ARGENTINA S.A.,Banco,202602,11,30546689979,30546689979,YPF SOCIEDAD ANONIMA,,NaN,...,0.0,2369287.0,0.0,0,0,0,0,0,0,0.0


Exportado df_situacion_1_top10 a la hoja 'df_situacion_1' en /content/drive/MyDrive/Tesis2026/all_situaciones_top10.xlsx

Top 10 de df_situacion_2 (ordenado por prestamos descendente):


,cod_entidad,nombre_entidad,grupo_entidad,fecha,tipo_id,nro_id,cuit,denominacion,marca_baja,cuit_reemplazo,...,contragar_pref_B,sin_contragar_pref,previsiones,deuda_cubierta,proceso_judicial,refinanciaciones,recat_obligatoria,sit_juridica,irrecuperable,dias_atraso
179805,00014,BANCO DE LA PROVINCIA DE BUENOS AIRES,Banco,202602,11,30507950848,30507950848,MOLINO CAÑUELAS SOCIEDAD ANONIMA COMERCIAL IND...,,NaN,...,0.0,0.0,0.0,0,0,0,0,0,0,0.0
153698,00011,BANCO DE LA NACION ARGENTINA,Banco,202602,11,30588855631,30588855631,SOCIEDAD ANONIMA LA SIBILA,,NaN,...,0.0,0.0,0.0,0,0,0,0,0,0,0.0
138104,00011,BANCO DE LA NACION ARGENTINA,Banco,202602,11,30517164530,30517164530,SIDECO AMERICANA S A,,NaN,...,0.0,0.0,0.0,0,0,0,0,0,0,0.0
141052,00011,BANCO DE LA NACION ARGENTINA,Banco,202602,11,30681491380,30681491380,DONTO SOCIEDAD ANONIMA,,NaN,...,0.0,0.0,0.0,0,0,0,0,0,0,0.0
390898,00285,BANCO MACRO S.A.,Banco,202602,11,30507950848,30507950848,MOLINO CAÑUELAS SOCIEDAD ANONIMA COMERCIAL IND...,,NaN,...,0.0,0.0,0.0,0,0,0,0,0,0,0.0
16605,00007,BANCO DE GALICIA Y BUENOS AIRES S.A.,Banco,202602,11,30507950848,30507950848,MOLINO CAÑUELAS SOCIEDAD ANONIMA COMERCIAL IND...,,NaN,...,0.0,0.0,0.0,0,0,0,0,0,0,0.0
266131,00029,BANCO DE LA CIUDAD DE BUENOS AIRES,Banco,202602,11,30507950848,30507950848,MOLINO CAÑUELAS SOCIEDAD ANONIMA COMERCIAL IND...,,NaN,...,0.0,0.0,0.0,0,0,0,0,0,0,0.0
6480,00007,BANCO DE GALICIA Y BUENOS AIRES S.A.,Banco,202602,11,30707633197,30707633197,SPEEDAGRO S.R.L.,,NaN,...,0.0,62.0,0.0,0,0,0,0,0,0,0.0
243596,00017,BANCO BBVA ARGENTINA S.A.,Banco,202602,11,30507950848,30507950848,MOLINO CAÑUELAS SOCIEDAD ANONIMA COMERCIAL IND...,,NaN,...,0.0,5.0,0.0,0,0,0,0,0,0,0.0
417405,00309,BANCO RIOJA SOCIEDAD ANONIMA UNIPERSONAL,Banco,202602,11,30682434720,30682434720,GENERACION MEDITERRANEA S.A.,,NaN,...,0.0,0.0,0.0,0,0,0,0,0,0,0.0


Exportado df_situacion_2_top10 a la hoja 'df_situacion_2' en /content/drive/MyDrive/Tesis2026/all_situaciones_top10.xlsx

Top 10 de df_situacion_3 (ordenado por prestamos descendente):


...

Exportado df_situacion_3_top10 a la hoja 'df_situacion_3' en /content/drive/MyDrive/Tesis2026/all_situaciones_top10.xlsx

Top 10 de df_situacion_4 (ordenado por prestamos descendente):


,cod_entidad,nombre_entidad,grupo_entidad,fecha,tipo_id,nro_id,cuit,denominacion,marca_baja,cuit_reemplazo,...,contragar_pref_B,sin_contragar_pref,previsiones,deuda_cubierta,proceso_judicial,refinanciaciones,recat_obligatoria,sit_juridica,irrecuperable,dias_atraso
34830,00007,BANCO DE GALICIA Y BUENOS AIRES S.A.,Banco,202602,11,30604456475,30604456475,LOS GROBO AGROPECUARIA S. A.,,NaN,...,0.0,0.0,0.0,0,0,0,0,0,0,0.0
158677,00011,BANCO DE LA NACION ARGENTINA,Banco,202602,11,30710557957,30710557957,ENTRE RIOS CRUSHING SOCIEDAD ANONIMA,,NaN,...,0.0,0.0,0.0,0,0,0,0,0,0,0.0
261716,00027,BANCO SUPERVIELLE S.A.,Banco,202602,11,30501657820,30501657820,BODEGA NORTON S.A.,,NaN,...,0.0,0.0,0.0,0,0,0,0,0,0,0.0
417574,00309,BANCO RIOJA SOCIEDAD ANONIMA UNIPERSONAL,Banco,202602,11,30710227728,30710227728,MAGGIORA S. A.,,NaN,...,0.0,0.0,0.0,0,0,0,0,0,0,0.0
163562,00011,BANCO DE LA NACION ARGENTINA,Banco,202602,11,30709585254,30709585254,LUSTRAMAX SRL,,NaN,...,0.0,0.0,0.0,0,0,0,0,0,0,0.0
154935,00011,BANCO DE LA NACION ARGENTINA,Banco,202602,11,30712156909,30712156909,REFI PAMPA S.A.,,NaN,...,0.0,0.0,0.0,0,0,0,0,0,0,0.0
185470,00014,BANCO DE LA PROVINCIA DE BUENOS AIRES,Banco,202602,11,30501466464,30501466464,IMPSA S.A,,NaN,...,0.0,0.0,0.0,0,0,0,0,0,0,0.0
138054,00011,BANCO DE LA NACION ARGENTINA,Banco,202602,11,30521958134,30521958134,CURTIEMBRE ARLEI S A,,NaN,...,0.0,0.0,0.0,0,0,0,0,0,0,0.0
20296,00007,BANCO DE GALICIA Y BUENOS AIRES S.A.,Banco,202602,11,30592724541,30592724541,AGROFINA S A,,NaN,...,0.0,0.0,0.0,0,0,0,0,0,0,0.0
264477,00029,BANCO DE LA CIUDAD DE BUENOS AIRES,Banco,202602,11,30501062150,30501062150,CELULOSA ARGENTINA S A,,NaN,...,0.0,0.0,0.0,0,0,0,0,0,0,0.0


Exportado df_situacion_4_top10 a la hoja 'df_situacion_4' en /content/drive/MyDrive/Tesis2026/all_situaciones_top10.xlsx

Top 10 de df_situacion_5 (ordenado por prestamos descendente):


,cod_entidad,nombre_entidad,grupo_entidad,fecha,tipo_id,nro_id,cuit,denominacion,marca_baja,cuit_reemplazo,...,contragar_pref_B,sin_contragar_pref,previsiones,deuda_cubierta,proceso_judicial,refinanciaciones,recat_obligatoria,sit_juridica,irrecuperable,dias_atraso
412938,00285,BANCO MACRO S.A.,Banco,202602,11,30709085111,30709085111,CAMPOCAM S. A.,,NaN,...,0.0,0.0,0.0,0,0,0,0,0,0,0.0
143012,00011,BANCO DE LA NACION ARGENTINA,Banco,202602,11,30710964013,30710964013,FIDEICOMISO FINANCIERO GAS II,,NaN,...,0.0,0.0,0.0,0,0,0,0,0,0,0.0
329835,00083,BANCO DEL CHUBUT S.A.,Banco,202602,11,30710739605,30710739605,CHOEL S.R.L.,,NaN,...,0.0,0.0,0.0,0,0,0,0,0,0,0.0
147765,00011,BANCO DE LA NACION ARGENTINA,Banco,202602,11,30559697857,30559697857,CURTIEMBRES FONSECA SA,,NaN,...,0.0,0.0,0.0,0,0,0,0,0,0,0.0
248717,00020,BANCO DE LA PROVINCIA DE CORDOBA S.A.,Banco,202602,11,33711054869,33711054869,LA REDENCION S. A.,,NaN,...,0.0,0.0,0.0,0,0,0,0,0,0,0.0
266318,00029,BANCO DE LA CIUDAD DE BUENOS AIRES,Banco,202602,11,30552587827,30552587827,COMPAÑIA ARGENTINA DE GRANOS S A,,NaN,...,0.0,0.0,0.0,0,0,0,0,0,0,0.0
330861,00083,BANCO DEL CHUBUT S.A.,Banco,202602,11,30710791461,30710791461,TRANSREDES SA,,NaN,...,0.0,0.0,0.0,0,0,0,0,0,0,0.0
266411,00029,BANCO DE LA CIUDAD DE BUENOS AIRES,Banco,202602,11,30715393251,30715393251,START.AR S. A.,,NaN,...,0.0,0.0,0.0,0,0,0,0,0,0,0.0
417500,00309,BANCO RIOJA SOCIEDAD ANONIMA UNIPERSONAL,Banco,202602,11,30716368544,30716368544,FIDEICOMISO FINANCIERO VICENTIN EXPORTACIONES ...,,NaN,...,0.0,0.0,0.0,0,0,0,0,0,0,0.0
232767,00017,BANCO BBVA ARGENTINA S.A.,Banco,202602,11,30707778217,30707778217,DEMISIONES S.R.L.,,NaN,...,0.0,0.0,0.0,0,0,1,0,0,0,192.0


Exportado df_situacion_5_top10 a la hoja 'df_situacion_5' en /content/drive/MyDrive/Tesis2026/all_situaciones_top10.xlsx

¡Proceso completado! Todos los dataframes top 10 han sido exportados a un único archivo Excel: /content/drive/MyDrive/Tesis2026/all_situaciones_top10.xlsx


In [8]:
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Assuming top_10_dfs is available from the previous cell's execution
# If not, you might need to re-run the previous cell (FpICUZJzQpgQ) first.

if not top_10_dfs:
    print("Warning: 'top_10_dfs' is empty or not defined. Please ensure the previous cell was run successfully.")
else:
    # Create a subplot figure with 3 rows and 2 columns
    # This will accommodate 5 plots, with one empty space.
    fig = make_subplots(rows=3, cols=2,
                        subplot_titles=[f'Top 10 - Situación {i}' for i in range(1, 6)],
                        vertical_spacing=0.15,
                        horizontal_spacing=0.08)

    row_idx = 1
    col_idx = 1

    for i in range(1, 6):
        df_situacion = top_10_dfs.get(f'df_situacion_{i}')

        if df_situacion is not None and not df_situacion.empty:
            # Create a bar chart for the current situacion
            bar_chart = go.Bar(
                x=df_situacion['denominacion'],
                y=df_situacion['prestamos'],
                name=f'Situación {i}',
                marker_color=px.colors.qualitative.Plotly[i-1] # Use different colors for each situacion
            )
            fig.add_trace(bar_chart, row=row_idx, col=col_idx)

            # Update layout for each subplot
            fig.update_xaxes(title_text='Denominación', row=row_idx, col=col_idx, tickangle=45, automargin=True)
            fig.update_yaxes(title_text='Préstamos', row=row_idx, col=col_idx, automargin=True)

        # Move to the next subplot position
        col_idx += 1
        if col_idx > 2:
            col_idx = 1
            row_idx += 1

    # Update overall layout
    fig.update_layout(
        title_text='Dashboard Interactivo: Top 10 Deudores Jurídicos por Situación',
        height=1200, # Adjust height as needed
        showlegend=False,
        hovermode='x unified'
    )

    fig.show()